# Diagnostic Test v2 - Correct Packing

Use Unsloth recommended approach:
- Use formatting_func instead of pre-processed text
- Enable packing correctly
- max_seq_length=512

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import time
import json
import sys

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

log_path = "results/diagnostic_v2_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Diagnostic v2 started")

In [ ]:
log("=== Data Pipeline ===")
t0 = time.time()

if not os.path.exists("data/splits/train.json"):
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

t1 = time.time()
log(f"Data pipeline: {t1-t0:.1f}s")

with open("data/splits/train.json") as f:
    train_data = json.load(f)
with open("data/splits/val.json") as f:
    val_data = json.load(f)

log(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
import torch

log("=== Model Loading ===")
log(f"GPU: {torch.cuda.get_device_name(0)}")

t0 = time.time()

from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

t1 = time.time()
log(f"Model loading: {t1-t0:.1f}s")
log(f"Max seq length: {MAX_SEQ_LENGTH}")

In [ ]:
log("=== LoRA Setup ===")
t0 = time.time()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

t1 = time.time()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"LoRA: {t1-t0:.1f}s, Trainable: {trainable:,}")

In [ ]:
log("=== Dataset Preparation ===")
t0 = time.time()

from datasets import Dataset

# Load raw data with conversations
def load_data(path):
    with open(path) as f:
        return json.load(f)

train_data = load_data("data/splits/train.json")
val_data = load_data("data/splits/val.json")

# Create dataset with messages format
def format_conversations(examples):
    texts = []
    for conversations in examples["conversations"]:
        messages = []
        for turn in conversations:
            role = "assistant" if turn["from"] == "gpt" else turn["from"]
            messages.append({"role": role, "content": turn["value"]})
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_dict({"conversations": [d["conversations"] for d in train_data]})
val_dataset = Dataset.from_dict({"conversations": [d["conversations"] for d in val_data]})

# Map to text format
train_dataset = train_dataset.map(format_conversations, batched=True, remove_columns=["conversations"])
val_dataset = val_dataset.map(format_conversations, batched=True, remove_columns=["conversations"])

t1 = time.time()
log(f"Dataset prep: {t1-t0:.1f}s")
log(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Check token lengths
lengths = [tokenizer(train_dataset[i]["text"], return_tensors="pt")["input_ids"].shape[1] for i in range(min(50, len(train_dataset)))]
log(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")

In [ ]:
log("=== Training Speed Test (10 steps) ===")

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs/diagnostic_v2",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=10,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

log("Starting 10-step training...")
t0 = time.time()
trainer.train()
t1 = time.time()

total_time = t1 - t0
time_per_step = total_time / 10

log(f"10 steps: {total_time:.1f}s")
log(f"Time/step: {time_per_step:.2f}s")
log(f"Baseline: 9.72s/step")
log(f"Speedup: {9.72 / time_per_step:.1f}x")

for entry in trainer.state.log_history:
    if "loss" in entry:
        log(f"Step {entry['step']}: loss={entry['loss']:.4f}")

In [ ]:
log("=" * 60)
log("DIAGNOSTIC SUMMARY v2")
log("=" * 60)
log(f"\nConfig: max_seq_length={MAX_SEQ_LENGTH}, packing=True")
log(f"Training speed: {time_per_step:.2f}s per step")
log(f"Baseline: 9.72s per step")
log(f"Speedup: {9.72 / time_per_step:.1f}x")

if time_per_step < 3:
    log("\n✅ PASS: Ready for full training")
elif time_per_step < 5:
    log("\n⚠️  ACCEPTABLE: Can proceed with caution")
else:
    log("\n❌ FAIL: Need further optimization")

log(f"\nLog: {log_path}")